# Test6 research — STU-Net inference explore

Explore [STU-Net](https://github.com/uni-medical/STU-Net) (scalable nnU-Net pretrained on TotalSegmentator) on a **few** RADCURE + HECKTOR CTs.

| Goal | This notebook |
|------|----------------|
| Download pretrained weights | yes (default **STU-Net-S**) |
| Inference (organs) | yes — 104 TotalSegmentator classes |
| Tumor (GTVp/GTVn) prediction | **no** — not in pretrained classes |
| Dice vs our GT labels | yes — **name-matched organs only** |
| Viz (CT / pred / GT+tumor) | yes |

**Tumor note:** STU-Net does not output GTVp/GTVn. We overlay **GT tumor** on figures for context only. Tumor Dice needs a later fine-tune on RADHECK.

See [`README.md`](README.md).


## 0. Paths & config

Edit the paths for your cluster. Defaults follow the `/media/HDD_8TB/xisca/...` layout used in Test4/5.


In [ ]:
from pathlib import Path
import os
import json
import shutil
import subprocess
import sys

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd

# --- edit these ---
TEST6_WORK_ROOT = Path(os.getenv(
    "TEST6_WORK_ROOT",
    "/media/HDD_8TB/xisca/work/research_test6_stunet",
))
REPO_ROOT = Path(os.getenv(
    "RADCURE_REPO",
    Path.cwd().resolve(),
))
# Prefer Dataset650 / Dataset152 nnUNet files (already spaced like training)
SAMPLE_CASES = [
    {
        "cohort": "radcure",
        "case_id": "RADCURE-0122",
        "stem": "case_0122",
        "image": Path(os.getenv(
            "TEST6_RADCURE_IMAGE",
            "/media/HDD_8TB/xisca/work/nnunet_radheck_test_1/Dataset650_TotalSegmentator/imagesTs/case_0122_0000.nii.gz",
        )),
        "label": Path(os.getenv(
            "TEST6_RADCURE_LABEL",
            "/media/HDD_8TB/xisca/work/nnunet_radheck_test_1/Dataset650_TotalSegmentator/labelsTs/case_0122.nii.gz",
        )),
    },
    {
        "cohort": "radcure",
        "case_id": "RADCURE-0040",
        "stem": "case_0040",
        "image": Path(os.getenv(
            "TEST6_RADCURE_IMAGE_2",
            "/media/HDD_8TB/xisca/work/nnunet_radheck_test_1/Dataset650_TotalSegmentator/imagesTs/case_0040_0000.nii.gz",
        )),
        "label": Path(os.getenv(
            "TEST6_RADCURE_LABEL_2",
            "/media/HDD_8TB/xisca/work/nnunet_radheck_test_1/Dataset650_TotalSegmentator/labelsTs/case_0040.nii.gz",
        )),
    },
    {
        "cohort": "hecktor",
        "case_id": "hecktor_sample",
        "stem": None,  # filled from filename
        "image": Path(os.getenv(
            "TEST6_HECKTOR_IMAGE",
            "/media/HDD_8TB/xisca/work/nnunet_hecktor_test1/Dataset152_TotalSegmentator/imagesTs",
        )),  # dir or file — resolved below
        "label": Path(os.getenv("TEST6_HECKTOR_LABEL", "")),
    },
]

STU_VARIANT = os.getenv("TEST6_STU_VARIANT", "small")  # small | base | large | huge
FAST_INFER = True  # --mode fast --disable_tta

# nnUNet / STU-Net install roots (cluster)
NNUNET_PATH = Path(os.getenv("NNUNET_PATH", "/media/HDD_8TB/xisca/code/nnUNet"))
STUNET_CLONE = TEST6_WORK_ROOT / "STU-Net"

ORGAN_DICT_OURS = REPO_ROOT / "image_processor" / "resources" / "organ_dictionary_hn_canonical.json"
LABEL_ORDERS = Path(__file__).resolve().parent / "label_orders.json" if "__file__" in dir() else Path("label_orders.json")
if not LABEL_ORDERS.is_file():
    LABEL_ORDERS = REPO_ROOT / "research_notebooks" / "test6_stunet" / "label_orders.json"

# Google Drive file IDs (TotalSegmentator-pretrained, 4k epochs)
GDRIVE = {
    "small": "1HReH6dDrEuXgHPrsw7OrHSjvEUF3f4mv",
    "base": "1BHCp1Ort-OaVFwaZmvsG4qHiKiPeNb4h",
    "large": "1KA1eXWWf_xAoJg5KHYrxTmfiz7wxGhHS",
    "huge": "1Qrq7oGPJ7ileFHWOAxwpeWdaB6hySptU",
}
CHK_NAME = {
    "small": "small_ep4k",
    "base": "base_ep4k",
    "large": "large_ep4k",
    "huge": "huge_ep4k",
}
TRAINER = {
    "small": "STUNetTrainer_small",
    "base": "STUNetTrainer_base",
    "large": "STUNetTrainer_large",
    "huge": "STUNetTrainer_huge",
}

for d in ("weights", "inputs", "predictions", "dice", "figures", "results_folder"):
    (TEST6_WORK_ROOT / d).mkdir(parents=True, exist_ok=True)

print("WORK:", TEST6_WORK_ROOT)
print("Variant:", STU_VARIANT, "→", TRAINER[STU_VARIANT], CHK_NAME[STU_VARIANT])


## 1. Resolve sample cases

If `TEST6_HECKTOR_IMAGE` is a folder, pick the first two `*.nii.gz` files.


In [ ]:
def resolve_samples(sample_cases):
    resolved = []
    for s in sample_cases:
        img = Path(s["image"])
        lab = Path(s["label"]) if s.get("label") else None
        if img.is_dir():
            files = sorted(img.glob("*_0000.nii.gz")) or sorted(img.glob("*.nii.gz"))
            take = files[:2]
            for f in take:
                stem = f.name.replace("_0000.nii.gz", "").replace(".nii.gz", "")
                lbl = None
                if lab and lab.is_dir():
                    cand = lab / f"{stem}.nii.gz"
                    lbl = cand if cand.is_file() else None
                elif lab and lab.is_file():
                    lbl = lab
                resolved.append({
                    "cohort": s["cohort"],
                    "case_id": f"{s['cohort']}:{stem}",
                    "stem": stem,
                    "image": f,
                    "label": lbl,
                })
        else:
            if not img.is_file():
                print("SKIP missing image:", img)
                continue
            stem = s.get("stem") or img.name.replace("_0000.nii.gz", "").replace(".nii.gz", "")
            lbl = lab if lab and lab.is_file() else None
            resolved.append({
                "cohort": s["cohort"],
                "case_id": s.get("case_id") or stem,
                "stem": stem,
                "image": img,
                "label": lbl,
            })
    return resolved

samples = resolve_samples(SAMPLE_CASES)
assert samples, "No sample images found — set TEST6_*_IMAGE paths"
for s in samples:
    print(f"{s['cohort']:8} {s['stem']:16} img={s['image'].name}  gt={'yes' if s['label'] else 'no'}")


## 2. Install / clone STU-Net helpers

Copies trainer + architecture modules into your nnUNet install (if not already present).  
Also clones the repo for `plan_files` + `label_orders`.


In [ ]:
# Lightweight deps for download
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])

if not STUNET_CLONE.is_dir():
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "https://github.com/uni-medical/STU-Net.git", str(STUNET_CLONE)]
    )
else:
    print("STU-Net clone exists:", STUNET_CLONE)

# Try to copy trainers into nnUNet v1 or v2 tree if present
def _copy_stunet_into_nnunet(nnunet_root: Path, stunet: Path) -> None:
    if not nnunet_root.is_dir():
        print("NNUNET_PATH not found — skip copy; ensure STU trainers are installed manually:", nnunet_root)
        return
    # nnUNet v1 layout
    v1_train = nnunet_root / "nnunet" / "training" / "network_training"
    v1_arch = nnunet_root / "nnunet" / "network_architecture"
    src_train = stunet / "nnUNet-1.7.1" / "nnunet" / "training" / "network_training"
    src_arch = stunet / "nnUNet-1.7.1" / "nnunet" / "network_architecture"
    # fallback: repo root network_* folders
    if not src_train.is_dir():
        src_train = stunet / "network_training"
    if not src_arch.is_dir():
        src_arch = stunet / "network_architecture"
    for src, dst in ((src_train, v1_train), (src_arch, v1_arch)):
        if src.is_dir() and dst.is_dir():
            for f in src.glob("STUNet*"):
                shutil.copy2(f, dst / f.name)
                print("copied", f.name, "→", dst)
        else:
            print("skip copy", src, "→", dst)

_copy_stunet_into_nnunet(NNUNET_PATH, STUNET_CLONE)
print("Done setup helpers")


## 3. Download weights + place under RESULTS_FOLDER

Layout expected by `nnUNet_predict` (Task 101):

```
RESULTS_FOLDER/nnUNet/3d_fullres/Task101_TotalSegmentator/
  STUNetTrainer_{variant}__nnUNetPlansv2.1/
    plans.pkl
    fold_0/{variant}_ep4k.model[.pkl]
```


In [ ]:
import gdown

results_root = TEST6_WORK_ROOT / "results_folder"
os.environ["RESULTS_FOLDER"] = str(results_root)
os.environ["nnUNet_results"] = str(results_root)  # v2-style if needed

task_dir = (
    results_root
    / "nnUNet"
    / "3d_fullres"
    / "Task101_TotalSegmentator"
    / f"{TRAINER[STU_VARIANT]}__nnUNetPlansv2.1"
)
fold_dir = task_dir / "fold_0"
fold_dir.mkdir(parents=True, exist_ok=True)

# plans.pkl from STU-Net repo
plan_src = STUNET_CLONE / "plan_files" / f"{TRAINER[STU_VARIANT]}__nnUNetPlansv2.1" / "plans.pkl"
if not plan_src.is_file():
    # some clones keep plans flat
    candidates = list((STUNET_CLONE / "plan_files").rglob("plans.pkl"))
    plan_src = next((p for p in candidates if STU_VARIANT in str(p)), None)
assert plan_src and plan_src.is_file(), f"plans.pkl not found under {STUNET_CLONE}/plan_files"
shutil.copy2(plan_src, task_dir / "plans.pkl")
print("plans:", task_dir / "plans.pkl")

model_path = fold_dir / f"{CHK_NAME[STU_VARIANT]}.model"
model_pkl = fold_dir / f"{CHK_NAME[STU_VARIANT]}.model.pkl"
if not model_path.is_file():
    print("Downloading STU-Net weights (Google Drive)…")
    # gdown may download a zip or the .model file depending on share type
    out = TEST6_WORK_ROOT / "weights" / f"{CHK_NAME[STU_VARIANT]}_download"
    out.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=GDRIVE[STU_VARIANT], output=str(out), quiet=False)
    # Heuristic: if zip, extract; if .model, move
    if out.suffix == ".zip" or out.is_file() and out.stat().st_size > 1_000_000:
        import zipfile
        try:
            with zipfile.ZipFile(out, "r") as zf:
                zf.extractall(TEST6_WORK_ROOT / "weights" / STU_VARIANT)
            found = list((TEST6_WORK_ROOT / "weights" / STU_VARIANT).rglob(f"{CHK_NAME[STU_VARIANT]}.model"))
            assert found, "Downloaded zip but .model not found"
            shutil.copy2(found[0], model_path)
            pkl = found[0].with_suffix(found[0].suffix + ".pkl")
            if not pkl.is_file():
                pkl = Path(str(found[0]) + ".pkl")
            if pkl.is_file():
                shutil.copy2(pkl, model_pkl)
        except zipfile.BadZipFile:
            shutil.copy2(out, model_path)
            # companion pkl often shared separately; predict may still work with plans
else:
    print("Weights already present:", model_path)

print("Ready:", model_path.exists(), model_path)


## 4. Stage inputs + run inference

Uses `nnUNet_predict` (v1 CLI). Requires GPU and STU trainers on `PYTHONPATH` / nnUNet install.


In [ ]:
input_dir = TEST6_WORK_ROOT / "inputs"
pred_dir = TEST6_WORK_ROOT / "predictions"
input_dir.mkdir(exist_ok=True)
pred_dir.mkdir(exist_ok=True)

# Fresh input folder with nnUNet naming
for f in input_dir.glob("*.nii.gz"):
    f.unlink()

for s in samples:
    dst = input_dir / f"{s['stem']}_0000.nii.gz"
    if not dst.exists():
        shutil.copy2(s["image"], dst)
    print("staged", dst.name)

cmd = [
    "nnUNet_predict",
    "-i", str(input_dir),
    "-o", str(pred_dir),
    "-t", "101",
    "-m", "3d_fullres",
    "-f", "0",
    "-tr", TRAINER[STU_VARIANT],
    "-chk", CHK_NAME[STU_VARIANT],
]
if FAST_INFER:
    cmd += ["--mode", "fast", "--disable_tta"]

print("Running:\n ", " ".join(cmd))
print("RESULTS_FOLDER=", os.environ.get("RESULTS_FOLDER"))
env = os.environ.copy()
env["RESULTS_FOLDER"] = str(results_root)
# Ensure repo nnUNet is preferred if set
if NNUNET_PATH.is_dir():
    env["PYTHONPATH"] = str(NNUNET_PATH) + os.pathsep + env.get("PYTHONPATH", "")

try:
    subprocess.check_call(cmd, env=env)
except FileNotFoundError:
    print(
        "nnUNet_predict not found on PATH.\n"
        "Activate the env where nnUNet is installed, or run the equivalent "
        "nnUNetv2 predict after placing STU trainers.\n"
        "See research_notebooks/test6_stunet/README.md"
    )
    raise


## 5. Load STU-Net label map + our organ dictionary

Dice is computed only for organs whose **names match** (after light normalisation).


In [ ]:
with open(LABEL_ORDERS) as f:
    stu_idx_to_name = {int(k): v for k, v in json.load(f).items()}

ours = {}
if ORGAN_DICT_OURS.is_file():
    with open(ORGAN_DICT_OURS) as f:
        ours = json.load(f)
ours_idx_to_name = {int(v): k for k, v in ours.items()}

def norm_name(n: str) -> str:
    return n.lower().replace("-", "_").replace(" ", "_")

stu_by_norm = {norm_name(v): k for k, v in stu_idx_to_name.items() if k > 0}
ours_by_norm = {norm_name(k): v for k, v in ours.items() if k not in (
    "background", "anatomical_region", "other-tissue", "GTVp", "GTVn"
)}

matched = sorted(set(stu_by_norm) & set(ours_by_norm))
print(f"STU classes: {len(stu_idx_to_name)-1}  Our organs: {len(ours_by_norm)}  Name matches: {len(matched)}")
print("Matched examples:", matched[:20])
print("Note: GTVp/GTVn are NOT in STU-Net pretrained classes.")


## 6. Dice (name-matched organs) + tumor GT presence check


In [ ]:
def dice_binary(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    denom = a.sum() + b.sum()
    if denom == 0:
        return 1.0 if inter == 0 else 0.0
    return float(2 * inter / denom)

rows = []
for s in samples:
    pred_path = pred_dir / f"{s['stem']}.nii.gz"
    if not pred_path.is_file():
        # nnUNet sometimes keeps _0000 stripped differently
        alts = list(pred_dir.glob(f"{s['stem']}*.nii.gz"))
        pred_path = alts[0] if alts else None
    if pred_path is None or not pred_path.is_file():
        print("Missing prediction for", s["stem"])
        continue

    pred = nib.load(str(pred_path)).get_fdata().astype(np.int32)
    gt = None
    if s["label"] and Path(s["label"]).is_file():
        gt = nib.load(str(s["label"])).get_fdata().astype(np.int32)
        if gt.shape != pred.shape:
            print(f"Shape mismatch {s['stem']}: gt {gt.shape} pred {pred.shape} — skip Dice")
            gt = None

    gtvp_idx = ours.get("GTVp")
    gtvn_idx = ours.get("GTVn")
    row = {
        "cohort": s["cohort"],
        "stem": s["stem"],
        "n_pred_labels": int(len(np.unique(pred)) - 1),
        "gtvp_voxels_gt": int(np.sum(gt == gtvp_idx)) if gt is not None and gtvp_idx else None,
        "gtvn_voxels_gt": int(np.sum(gt == gtvn_idx)) if gt is not None and gtvn_idx else None,
        "tumor_dice_stu": None,  # intentionally N/A
    }

    if gt is not None:
        dices = []
        for name in matched:
            si, oi = stu_by_norm[name], ours_by_norm[name]
            d = dice_binary(gt == oi, pred == si)
            row[f"dice_{name}"] = d
            dices.append(d)
        row["dice_matched_mean"] = float(np.mean(dices)) if dices else np.nan
    else:
        row["dice_matched_mean"] = np.nan

    rows.append(row)
    print(
        f"{s['stem']}: pred_labels={row['n_pred_labels']}  "
        f"matched_mean_dice={row['dice_matched_mean']}"
    )

df = pd.DataFrame(rows)
csv_path = TEST6_WORK_ROOT / "dice" / "stunet_sample_dice.csv"
df.to_csv(csv_path, index=False)
display(df[[c for c in df.columns if not c.startswith("dice_") or c == "dice_matched_mean"]])
print("saved", csv_path)


## 7. Visualisation — CT | STU-Net organs | GT (+ tumor)

GTVp = red, GTVn = pink when present in GT. STU-Net organs use a fixed palette (no red/pink).


In [ ]:
from matplotlib.colors import ListedColormap

def window_ct(vol, z, p=(1, 99)):
    sl = vol[:, :, z]
    lo, hi = np.percentile(sl, p)
    x = (sl - lo) / (hi - lo + 1e-8)
    return np.clip(x, 0, 1)

def overlay_labels(ax, base, lab, colors, alpha=0.45, highlight=None):
    ax.imshow(base.T, cmap="gray", origin="lower")
    rgba = np.zeros((*lab.shape, 4), dtype=np.float32)
    for idx, rgb in colors.items():
        if idx == 0:
            continue
        rgba[lab == idx] = (*rgb, alpha)
    if highlight:
        for idx, rgb in highlight.items():
            rgba[lab == idx] = (*rgb, 0.85)
    ax.imshow(np.transpose(rgba, (1, 0, 2)), origin="lower")
    ax.axis("off")

# simple deterministic colours for STU labels (skip red/pink-ish)
rng = np.random.default_rng(0)
stu_colors = {}
for i in range(1, 105):
    rgb = rng.random(3)
    if rgb[0] > 0.7 and rgb[1] < 0.45:
        rgb[0] = 0.2
    stu_colors[i] = tuple(rgb.tolist())

gt_colors = {i: (0.3, 0.7, 0.9) for i in range(1, 100)}
tumor_hi = {}
if ours.get("GTVp") is not None:
    tumor_hi[ours["GTVp"]] = (1.0, 0.0, 0.0)
if ours.get("GTVn") is not None:
    tumor_hi[ours["GTVn"]] = (1.0, 0.41, 0.71)

for s in samples:
    pred_path = pred_dir / f"{s['stem']}.nii.gz"
    if not pred_path.is_file():
        alts = list(pred_dir.glob(f"{s['stem']}*.nii.gz"))
        if not alts:
            continue
        pred_path = alts[0]
    ct = nib.load(str(s["image"])).get_fdata().astype(np.float32)
    pred = nib.load(str(pred_path)).get_fdata().astype(np.int32)
    gt = nib.load(str(s["label"])).get_fdata().astype(np.int32) if s["label"] and Path(s["label"]).is_file() else None

    z = ct.shape[2] // 2
    fig, axes = plt.subplots(1, 3 if gt is not None else 2, figsize=(12, 4))
    base = window_ct(ct, z)
    axes[0].imshow(base.T, cmap="gray", origin="lower")
    axes[0].set_title(f"CT  z={z}")
    axes[0].axis("off")
    overlay_labels(axes[1], base, pred[:, :, z] if pred.ndim == 3 else pred, stu_colors)
    axes[1].set_title("STU-Net organs")
    if gt is not None:
        overlay_labels(axes[2], base, gt[:, :, z], gt_colors, highlight=tumor_hi)
        axes[2].set_title("Our GT (+ tumor)")
    fig.suptitle(f"{s['cohort']} | {s['stem']} | STU-Net-{STU_VARIANT}")
    out = TEST6_WORK_ROOT / "figures" / f"{s['stem']}_stunet_explore.png"
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print("saved", out)


## 8. Takeaways / next

- Pretrained STU-Net is an **organ** foundation model (104 TS classes), not a H&N tumor model.
- Useful next research steps:
  1. More samples / STU-Net-B if S looks promising on matched organs  
  2. Map STU-Net organs → our H&N set more carefully (synonyms)  
  3. Fine-tune on Dataset650 with GTVp/GTVn for a true Test6 vs Test4/5  

When ready for a full experiment, promote this into `pipelines/` + `experiments/registry.yaml` (`status: running`).
